# DSCI 6883 - Descriptive Analytics & Data Integrity
## Discussion 2.5 & 2.6 Follow-Up: Standard Deviation, Plain Numbers, and Methods

**Fall 2026 | Module 2 | Fernando Rubio Garcia**

In [0]:
# Housekeeping - run this once, then forget about it.
#
# A few cells below are *designed* to raise an error, so you can see what that error
# actually looks like. By default Jupyter stops a "Run All" at the first error it hits,
# which would stop this notebook partway through. The line below turns that off.
#
# It is wrapped in try/except because it only applies to Jupyter. If you are running
# somewhere else (Databricks, a plain .py script), it does nothing and that is fine.
try:
    get_ipython().kernel._abort_queues = lambda *a, **kw: None
except Exception:
    pass

### Where this came from

In the 2.5 & 2.6 discussion I asked which parts of the visualization videos needed more explanation. The answer was clear: the charts landed. Nobody was stuck on making a histogram or a box plot, so I am not going to re-record that. (Choosing the right chart and making it readable is Lecture 6.)

What did not land was one number. Matei asked it first: *"I'm not sure how to tell whether the value is high or low for a particular dataset."* Scott asked for rules of thumb and for how it is really used, and Jacob described the one place he does use it. So **Sections 1 to 4 are all standard deviation**, and they lean on the histogram and the box plot you just learned, because those two charts are how you check whether the number can be trusted.

Two more questions came in, and they are not about charts at all, which is fine. Thomas asked why his results print as `np.float64(22.588136)` instead of plain numbers (Section 5). Kara asked about functions versus methods, and `sorted()` versus `.sort()` (Section 6).

Jacob also described how he studies: follow along with the video, then turn the video off and do it again alone. That is exactly the right habit, and I recommend it to everyone. The second pass is where you find out what you actually know.

If your part is in here and it still does not click, reply in the discussion with the **section number and the cell** where it stopped making sense.

### How to use this notebook

1. **Run every cell yourself**, top to bottom, with `Shift + Enter`.
2. **Predict the output before you run a cell.**
3. **Do the `Your Turn` exercises** before opening the answers.
4. Cells marked `SUPPOSED TO FAIL` are meant to raise an error. Read the error, then keep going.
5. The **assignment is at the end**. It uses the real Superstore file from 2.6.

Everything in this notebook is generated in memory. It does not read or write any files, so it runs anywhere, including Databricks.

### Outline

| Section | Topic | From the post by |
|---|---|---|
| 1 | What a standard deviation measures | Matei |
| 2 | "High or low?" Compared to what | Matei, Scott |
| 3 | Rules of thumb, and when they break | Scott |
| 4 | How it is really used | Scott, Jacob |
| 5 | Plain numbers instead of `np.float64(...)` | Thomas |
| 6 | Functions vs methods, `sorted()` vs `.sort()` | Kara, Matei |
| | **Assignment** | |

Section 2 is the direct answer to Matei and Scott. Section 4 is the one I most want you to take into your project, because it is where standard deviation meets data integrity.

---
## Setup: build the practice data

Run the next cell once. It builds three small datasets: delivery times for two carriers, a set of exam scores, and an orders table shaped like Superstore (most orders small, a few very large).

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("pandas version:", pd.__version__, "| numpy version:", np.__version__)

rng = np.random.default_rng(6883)

# Delivery times, in days, for two carriers. About the same average, very different spread.
deliveries = pd.DataFrame({
    "carrier": ["A"] * 500 + ["B"] * 500,
    "days":    np.concatenate([rng.normal(5, 0.5, 500), rng.gamma(4, 1.25, 500)]).round(1),
})

# Exam scores: roughly bell-shaped.
scores = pd.Series(rng.normal(75, 8, 2000).round(1), name="score")

# Orders, shaped like Superstore sales: skewed to the right.
subcats = ["Binders", "Chairs", "Copiers", "Labels", "Paper", "Phones", "Storage", "Tables"]
orders = pd.DataFrame({
    "sub_category": rng.choice(subcats, 5000),
    "sales":        rng.lognormal(4.4, 1.2, 5000).round(2),
})

print("deliveries:", deliveries.shape, "  scores:", scores.shape, "  orders:", orders.shape)

In [0]:
deliveries.head(5)

In [0]:
scores.head(5)

In [0]:
orders.head(5)

---
# 1. What a standard deviation measures

**From Matei's post:** *"I would like more explanation on how to interpret standard deviation. I'm not sure how to tell whether the value is high or low for a particular dataset and what that tells us about how spread out the data is."*

**From Scott's reply:** *"I've worked with data and numbers a lot and I somehow have never gotten comfortable with what it is and how to use it intuitively."*

Start with what the number is, in one sentence: **the standard deviation is the typical distance between a value and the mean.** Here are two tiny datasets with the same mean and very different spread.

In [0]:
tight = pd.Series([48, 49, 50, 51, 52])
wide  = pd.Series([10, 30, 50, 70, 90])

print("mean of tight:", tight.mean(), "  mean of wide:", wide.mean())

Both average 50. The mean cannot tell them apart. Now do the calculation by hand on `wide`, one column per step:

In [0]:
steps = pd.DataFrame({"value": wide})
steps["distance from mean"] = steps["value"] - steps["value"].mean()
steps["squared"] = steps["distance from mean"] ** 2
steps

In [0]:
variance = steps["squared"].sum() / (len(steps) - 1)

print("average squared distance (the variance):", variance)
print("square root of that (the std):          ", round(variance ** 0.5, 2))
print()
print("pandas, wide.std(): ", round(wide.std(), 2))
print("pandas, tight.std():", round(tight.std(), 2))

Three things to take from that:

- The distances are squared so the negatives do not cancel the positives, then the square root brings the answer back to the original units. Squaring also means **far-away values count more than close ones**. Remember that for Section 4.
- `wide` has a std of about 32 and `tight` has about 1.6. Same mean, twenty times the spread. Read the first one as "a typical value in `wide` is about 30 away from 50."
- **The std is in the same units as the data.** A std of delivery times is in days. A std of sales is in dollars. That is the whole reason Matei's question is hard: 32 is not high or low. It is 32 *of something*, and you have to compare it to something else in the same units.

(Why divide by `len - 1` and not `len`? Section 4.5.)

---
# 2. "High or low?" Compared to what

**From Scott's post:** *"The calculation is straightforward. I know the definitions. But I'm always curious about how it's really used. How do you know when looking at a standard deviation value if it's high or low? Do you have any rules of thumb for interpreting it?"*

Short answer: **a standard deviation on its own is never high or low.** It becomes high or low when you put it next to one of three things. I use them in this order.

### 2.1 Compared to the mean

Divide the std by the mean. The result is called the **coefficient of variation** (CV), and it reads as "the typical value is this far from the mean, as a share of the mean."

In [0]:
summary = deliveries.groupby("carrier")["days"].agg(["mean", "std"])
summary["cv"] = summary["std"] / summary["mean"]
summary.round(2)

Both carriers average about 5 days. For carrier A, a typical delivery is within about 10% of that. For carrier B it is about 50% off.

Here are the rules of thumb Scott asked for. They are mine, not a law, and they only apply to quantities that cannot go below zero (time, weight, price, counts):

| CV (std / mean) | How I read it |
|---|---|
| under 10% | Tight. The mean describes almost every value. |
| 10% to 30% | Moderate. The mean is a fair summary. |
| 30% to 100% | Wide. Report the spread next to the mean, always. |
| over 100% | The std is bigger than the mean. The data is skewed, and the mean is not a typical value. Go to Section 3.2. |

Do not use the CV on a column that crosses zero. Profit can be negative, so its mean can be close to zero, and dividing by a number close to zero gives nonsense. The same goes for temperatures in Fahrenheit or Celsius.

### 2.2 Compared to another group

This is the most common use in practice, and it needs no rule of thumb at all. You are not asking "is 2.5 days a lot?" You are asking "which carrier is more predictable?" Look at the two histograms, drawn on the same axis.

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharex=True, sharey=True)

for ax, name in zip(axes, ["A", "B"]):
    days = deliveries.loc[deliveries["carrier"] == name, "days"]
    ax.hist(days, bins=30)
    ax.axvline(days.mean(), color="black", linestyle="--")
    ax.set_title(f"Carrier {name}: mean {days.mean():.1f} days, std {days.std():.1f}")
    ax.set_xlabel("Delivery time (days)")

axes[0].set_ylabel("Number of deliveries")
plt.tight_layout()
plt.show()

The dashed line is the mean, and it is in about the same place on both. A report that only gives the mean says these two carriers are the same. **The std is the number that says what the histogram shows**: the width of the pile.

That is what "how spread out the data is" means, Matei. A small std is a narrow histogram, where the mean is a good guess for any single value. A large std is a wide one, where the mean is a poor guess.

### 2.3 Compared to a line that matters

Suppose you promise customers delivery within 7 days. Now the question is not about the std at all. It is "how often do we break the promise?"

In [0]:
deliveries["late"] = deliveries["days"] > 7

(deliveries.groupby("carrier")["late"].mean() * 100).round(1)

Carrier A is essentially never late. Carrier B breaks the promise about one time in five, with the same average.

This is the answer I would give a manager who asked "is that standard deviation high?" **It is high if it pushes you over a line you care about.** The line can be a promise, a budget, a tolerance on a part, or a passing grade. If there is no such line and no other group to compare to, fall back on the CV.

So the three comparisons, in the order I reach for them:

| Compare the std to | The question it answers |
|---|---|
| another group, or the same group last year | Which one is more consistent? |
| a line that matters | How often do we cross it? |
| the mean (the CV) | How good a summary is the mean? |

#### Your Turn 2.1 - Change the units

Make a new column with the delivery time in **hours** instead of days. For carrier B, compute the mean, the std, and the CV in hours. Before you run it: which of the three numbers will change, and which will not?

In [0]:
# Your code here

In [0]:
deliveries["hours"] = deliveries["days"] * 24

b = deliveries[deliveries["carrier"] == "B"]
print(f"days:  mean {b['days'].mean():.2f}, std {b['days'].std():.2f}, cv {b['days'].std() / b['days'].mean():.2f}")
print(f"hours: mean {b['hours'].mean():.2f}, std {b['hours'].std():.2f}, cv {b['hours'].std() / b['hours'].mean():.2f}")

<details>
<summary><b>Show answer</b></summary>

```python
deliveries["hours"] = deliveries["days"] * 24

b = deliveries[deliveries["carrier"] == "B"]
print(f"days:  mean {b['days'].mean():.2f}, std {b['days'].std():.2f}, cv {b['days'].std() / b['days'].mean():.2f}")
print(f"hours: mean {b['hours'].mean():.2f}, std {b['hours'].std():.2f}, cv {b['hours'].std() / b['hours'].mean():.2f}")
```

The mean and the std are both 24 times bigger. The CV does not move.

Nothing about the deliveries changed, and the std went from about 2.5 to about 60. That is the proof that the std alone cannot be "high": I can make it any size I like by changing the units. The CV, the comparison between carriers, and the share of late deliveries all stay the same, which is why those are the three things to look at.

</details>

---
# 3. Rules of thumb, and when they break

There is one famous rule of thumb for standard deviation, and one thing nobody tells you about it.

### 3.1 The 68-95-99.7 rule

For data shaped like a bell (one peak in the middle, falling off evenly on both sides):

- about **68%** of values are within 1 std of the mean,
- about **95%** are within 2,
- about **99.7%** are within 3.

That is the rule that makes a std useful. If exam scores have a mean of 75 and a std of 8, then about two thirds of the class is between 67 and 83, and almost everybody is between 59 and 91. Check it on the exam scores:

In [0]:
def share_within(series, k):
    'Share of the values that are within k standard deviations of the mean.'
    mean, std = series.mean(), series.std()
    return series.between(mean - k * std, mean + k * std).mean()


def rule_check(series):
    'Compare a column against the 68-95-99.7 rule.'
    return pd.DataFrame({
        "within":          ["1 std", "2 std", "3 std"],
        "bell-curve rule": [0.68, 0.95, 0.997],
        "this column":     [round(share_within(series, k), 3) for k in (1, 2, 3)],
    })


print(f"scores: mean {scores.mean():.1f}, std {scores.std():.1f}")
rule_check(scores)

In [0]:
scores.info()

Very close. On bell-shaped data, "mean plus or minus 2 std" is a good description of where nearly everything is.

### 3.2 What nobody tells you: it only works on bell-shaped data

Now the same check on `sales`, which is shaped like the real Superstore column.

In [0]:
mean, median, std = orders["sales"].mean(), orders["sales"].median(), orders["sales"].std()

print(f"mean {mean:,.2f} | median {median:,.2f} | std {std:,.2f}")
print(f"mean - 1 std = {mean - std:,.2f}")
print(f"mean + 1 std = {mean + std:,.2f}")

rule_check(orders["sales"])

Read those lines one at a time, because every one of them is a warning sign:

- **The std is bigger than the mean.** The CV is over 100%.
- **The mean is about twice the median.** A few very large orders are pulling the mean up.
- **"Mean minus 1 std" is a negative number.** The rule says two thirds of orders are between a negative sale and about 500 dollars. No order is negative.
- **Over 90% of orders are within 1 std**, not 68%. The rule is simply wrong here.

The histograms show why. One of these is a bell, and one is not.

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

panels = [(scores, "Exam score", None),
          (orders["sales"], "Sales in dollars (orders under 1,000 only)", (0, 1000))]

for ax, (series, label, limits) in zip(axes, panels):
    ax.hist(series, bins=40, range=limits)
    ax.axvline(series.mean(), color="black", linestyle="--", label="mean")
    ax.axvline(series.median(), color="black", linestyle=":", label="median")
    ax.set_xlabel(label)
    ax.set_ylabel("Count")
    ax.legend()

axes[0].set_title("Bell-shaped: mean and std describe it")
axes[1].set_title("Skewed: they do not")
plt.tight_layout()
plt.show()

The right-hand chart stops at 1,000 dollars. About 2% of the orders are larger than that, one of them over 13,000, and including them would squash everything else into a single bar. That long thin tail is what "skewed" means, and it is what pulls the mean (dashed) so far to the right of the median (dotted).

The std is not *wrong* on the right-hand chart. It is correctly reporting that the typical squared distance from the mean is large. But it was built for symmetric data, and on a skewed column it is describing a shape that is not there.

**This is why videos 85-90 teach the histogram and the std together. Draw the histogram first. It tells you whether the std means anything.**

### 3.3 What to report instead

For skewed data, use the two numbers the box plot is drawn from: the **median** (the line in the box) and the **interquartile range**, or IQR (the height of the box, from the 25th to the 75th percentile).

In [0]:
q1, q3 = orders["sales"].quantile([0.25, 0.75])

print(f"median {median:,.2f}")
print(f"the middle half of orders is between {q1:,.2f} and {q3:,.2f} (IQR {q3 - q1:,.2f})")

"A typical order is about 80 dollars, and half of all orders are between about 35 and 185" is a sentence a manager can use, and every number in it is a real order size. "Orders average 170 plus or minus 340" is not.

| The histogram looks | Report | Chart |
|---|---|---|
| roughly symmetric, one peak | mean and std | histogram |
| skewed, or with a few extreme values | median and IQR | box plot, or a histogram |
| two peaks | neither. You have two groups mixed together. Split them first. | one histogram per group |

A quick test without drawing anything: **if the mean and the median are far apart, or the std is bigger than the mean on a column that cannot be negative, the data is skewed.** `.describe()` prints all of those at once.

> **One rule that works on any shape.** No matter how the data is distributed, at least 75% of the values are within 2 std of the mean, and at least 89% are within 3. It is much weaker than 68-95-99.7, but it is never wrong, and it is the reason "more than 3 std from the mean" is always at least a little unusual.

#### Your Turn 3.1 - One typo

Someone types a score of `7500` where they meant `75`. The cell below makes a copy of `scores` with that one typo in it. Compute the mean, std, median, and IQR for `scores` and for `scores_typo`. Out of 2,000 values, one changed. Which of the four numbers moved?

In [0]:
scores_typo = scores.copy()
scores_typo.iloc[0] = 7500

# Your code here

In [0]:
def four_numbers(series):
    q1, q3 = series.quantile([0.25, 0.75])
    return {"mean": series.mean(), "std": series.std(), "median": series.median(), "IQR": q3 - q1}

pd.DataFrame({"scores": four_numbers(scores), "scores_typo": four_numbers(scores_typo)}).round(2)

<details>
<summary><b>Show answer</b></summary>

```python
def four_numbers(series):
    q1, q3 = series.quantile([0.25, 0.75])
    return {"mean": series.mean(), "std": series.std(), "median": series.median(), "IQR": q3 - q1}

pd.DataFrame({"scores": four_numbers(scores), "scores_typo": four_numbers(scores_typo)}).round(2)
```

The mean moves by a few points. The **std goes from about 8 to about 166**, twenty times bigger, because of one row in two thousand. The median moves by five hundredths of a point, and the IQR does not move at all.

That is the squaring from Section 1: a value 7,400 away from the mean contributes 7,400 squared. The std is the statistic most easily wrecked by one bad value, which makes it both a good alarm (Section 4.2) and a bad ruler (Section 4.3).

</details>

---
# 4. How it is really used

**From Jacob's reply to Scott:** *"Personally, the only times I've used standard deviation stems from filtering out absurdly high numbers (via typos by other people, usually) that would end up throwing off the rest of my calculations."*

Jacob, that is the most common real use there is, and it has a trap in it that this section is mostly about. First, the tool underneath it.

### 4.1 The z-score: how many std from the mean

Take a value, subtract the mean, divide by the std. The result is the **z-score**: how far the value is from the mean, counted in standard deviations. It removes the units, so you can compare things that were measured on different scales.

You scored 82 on two exams. Which was the better result?

In [0]:
def z_score(value, mean, std):
    return (value - mean) / std

print("Exam 1 (class mean 75, std 8):", z_score(82, 75, 8))
print("Exam 2 (class mean 70, std 4):", z_score(82, 70, 4))

The same 82 is a little above average on exam 1 (under 1 std) and exceptional on exam 2 (3 std, which the rule from 3.1 says almost nobody reaches). **This is the std doing its job: it turns "82" into "how unusual is 82, here?"**

### 4.2 Flagging unusual values

Jacob's use is the z-score applied to a whole column: compute z for every row, and look at the rows where it is beyond 3 or below -3. On the exam scores with the typo from Your Turn 3.1:

In [0]:
z = (scores_typo - scores_typo.mean()) / scores_typo.std()

print("rows with |z| > 3:")
print(pd.DataFrame({"score": scores_typo, "z": z.round(1)})[z.abs() > 3])

The typo has a z-score in the forties. Found it. With 2,000 rows, one bad value cannot hide.

### 4.3 The trap: the typo bends the ruler

Now a small table, which is where typos usually get looked for. Ten invoice amounts, and one of them has an extra pair of zeros.

In [0]:
invoices = pd.Series([120, 135, 128, 142, 131, 125, 138, 13300, 129, 133], name="amount")

z = (invoices - invoices.mean()) / invoices.std()

print(f"mean {invoices.mean():,.1f}   std {invoices.std():,.1f}")
pd.DataFrame({"amount": invoices, "z": z.round(2)})

The invoice for 13,300 is a hundred times bigger than the others, and its z-score is **under 3**. The "more than 3 std" filter does not flag it.

Look at the mean and the std printed above the table. The typo dragged the mean up to over a thousand and blew the std up to over four thousand. **The value you are trying to catch is part of the mean and the std you are measuring it against**, and in a small sample it is a big part. With ten values, no value can ever have a z-score above 2.85, however extreme it is. A z-score filter on a small table is a filter that cannot fire.

This is the same problem as Your Turn 3.1 from the other side: the std is easily wrecked by one bad value, so it is a poor ruler for finding bad values.

### 4.4 The box plot rule

The fix is to measure with something a typo cannot bend: the quartiles. This is the rule the box plot uses to decide which points to draw as dots.

- Lower fence: Q1 minus 1.5 times the IQR
- Upper fence: Q3 plus 1.5 times the IQR
- Anything outside the fences is flagged.

In [0]:
def iqr_fences(series):
    'The box plot fences: 1.5 IQR below the first quartile and above the third.'
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr


low, high = iqr_fences(invoices)

print(f"fences: {low:.2f} to {high:.2f}")
print("flagged:", invoices[(invoices < low) | (invoices > high)].tolist())

The fences sit just outside the ordinary invoices, and the typo is far beyond them. The quartiles are the same whether the largest value is 142 or 13,300, so the ruler did not move.

Here is the same rule as a picture, on the delivery times. **The dots past the whiskers are exactly the values outside these fences.** That is all they are.

In [0]:
fig, ax = plt.subplots(figsize=(6, 3.5))

ax.boxplot([deliveries.loc[deliveries["carrier"] == "A", "days"],
            deliveries.loc[deliveries["carrier"] == "B", "days"]])
ax.set_xticklabels(["Carrier A", "Carrier B"])
ax.set_ylabel("Delivery time (days)")
ax.set_title("The dots are the deliveries outside the 1.5 IQR fences")
plt.show()

#### Your Turn 4.1 - Two rules, one skewed column

On `orders["sales"]`, count how many rows each rule flags: `|z| > 3`, and outside the box plot fences (use `iqr_fences`). Then look at the five largest sales. Are they typos?

In [0]:
# Your code here

sales = orders["sales"]

z = (sales - sales.mean()) / sales.std()
low, high = iqr_fences(sales)

print("flagged by |z| > 3:      ", (z.abs() > 3).sum())
print("flagged by the IQR fences:", ((sales < low) | (sales > high)).sum())
print()
print(sales.sort_values(ascending=False).head())

<details>
<summary><b>Show answer</b></summary>

```python
sales = orders["sales"]

z = (sales - sales.mean()) / sales.std()
low, high = iqr_fences(sales)

print("flagged by |z| > 3:      ", (z.abs() > 3).sum())
print("flagged by the IQR fences:", ((sales < low) | (sales > high)).sum())
print()
print(sales.sort_values(ascending=False).head())
```

The z rule flags a little over 1% of the orders and the fences flag seven times as many, about 9%. Neither set is typos. This is a skewed column, and large orders are a normal part of it. A box plot of `sales` shows a long column of dots above the top whisker, and every one of them is a real order.

So the two rules disagree, and both are "wrong" in the sense that neither found an error. Which brings us to the part that matters.

</details>

### 4.5 A flag is a question, not a verdict

Both rules answer "which values are far from the rest?" Neither can answer "is this value wrong?" That one is yours, and it is a data integrity decision like every other one in this course.

| What you find when you look | What it is | What to do |
|---|---|---|
| Impossible: a negative quantity, an age of 200, a score of 7,500 out of 100 | An error | Fix it if you can find the true value. Otherwise blank it. Either way, put it in the decision log. |
| Possible but extreme: one very large order | A real value | Keep it. Summarize with the median, or report the result with and without it. |
| You cannot tell | Unknown | Keep it, flag it, and say so. Ask whoever owns the data. |

**Never delete a row only because a rule flagged it.** On the Superstore data that would throw away the largest orders, which are the ones the business cares about most. Jacob's case is the first row of the table: he knew the values were typos because he knew the data. The rule only told him where to look.

### 4.6 Other places you will meet it

Scott asked how it is really used. Beyond comparing groups (2.2) and flagging values (4.2), these are the ones you will run into:

- **"Plus or minus" in a report.** "Average delivery 5.0 days, plus or minus 0.5" is a mean and a std. Check which one they mean, because some reports use the standard error instead (the std divided by the square root of the number of rows), and that is a much smaller number.
- **Control limits.** Manufacturing and operations dashboards draw lines at the mean plus and minus 3 std, and investigate anything outside them. That is 4.2 as a chart.
- **Volatility.** In finance, the risk of an investment is reported as the std of its returns. Two funds with the same average return and different stds are carriers A and B.
- **Putting columns on one scale.** Machine learning methods often convert every column to z-scores first, so that a column measured in dollars does not outweigh a column measured in years.

### 4.7 Why your number may not match someone else's

One last practical thing. pandas and NumPy do not compute the same standard deviation by default.

In [0]:
values = wide.to_numpy()

print("pandas, wide.std():       ", round(wide.std(), 2), " (divides by n - 1)")
print("NumPy, np.std(values):    ", round(np.std(values), 2), " (divides by n)")
print("NumPy, np.std(.., ddof=1):", round(np.std(values, ddof=1), 2), " (now they agree)")

pandas divides by `n - 1`, which is the right choice when your rows are a sample from something bigger, and that is nearly always. NumPy divides by `n`. Excel has both: `STDEV.S` matches pandas and `STDEV.P` matches NumPy.

With thousands of rows the two are almost identical. With five rows they are not. If your std does not match a classmate's or a spreadsheet's, this is the first thing to check. In this course, use the pandas default.

---
# 5. Plain numbers instead of `np.float64(...)`

**From Thomas's post:** *"An issue I am running into is when I do calculations such as min, max, mean, std, etc., my outputs are in the format of: `(np.float64(22.588136), np.float64(3.8296986886051494))`. Is there a way I can get it to produce just numbers?"*

Yes, three ways. First, what is going on. Run this:

In [0]:
scores.mean(), scores.std()

If you are on NumPy 2 (the version line under Setup tells you), you see what Thomas sees. If you are on NumPy 1, you see two plain numbers, and you can skim this section.

**Nothing is wrong with the numbers.** `np.float64(75.2)` is 75.2. It adds, rounds, and compares like 75.2. Starting with NumPy 2, a NumPy number shows its type when it is displayed *inside* something else: a tuple, a list, or a dictionary. Two values separated by a comma are a tuple, so the label appears. This also came up in the 1.6 follow-up, for a dictionary of averages.

`print` of a single value shows it plain. `print` of a tuple does not:

In [0]:
print(scores.mean())
print((scores.mean(), scores.std()))

### 5.1 Fix 1: an f-string, which is what I use

The label is not the only problem with that output. `3.8296986886051494` has sixteen digits, and nobody wants those either. An f-string fixes both at once, and lets you say what each number is.

In [0]:
mean, std = scores.mean(), scores.std()

print(f"mean {mean:.2f}, std {std:.2f}")
print(f"lowest {scores.min():.1f}, highest {scores.max():.1f}")

The part after the colon is the format. These four cover almost everything:

| Code | Meaning | `1234.5678` becomes |
|---|---|---|
| `:.2f` | 2 decimals | `1234.57` |
| `:,.2f` | 2 decimals, with thousands commas | `1,234.57` |
| `:,.0f` | no decimals, with commas | `1,235` |
| `:.1%` | a share as a percent (`0.256` becomes `25.6%`) | |

### 5.2 Fix 2: ask pandas for a table instead of a tuple

If you want min, max, mean, and std together, do not compute them one by one. Ask for all of them in one call, and pandas hands back a labelled table.

In [0]:
scores.agg(["min", "max", "mean", "std"]).round(2)

In [0]:
orders.groupby("sub_category")["sales"].agg(["min", "max", "mean", "std"]).round(2)

No labels to remove, the numbers are named, and it works per group with one more word. For a whole DataFrame, `.describe().round(2)` gives you a similar table for every numeric column.

### 5.3 Fix 3: convert to a plain Python number

When you really need a plain number, for example to put in a dictionary or to write to a file, wrap it in `float()` (or `int()` for a count).

In [0]:
print(type(scores.mean()), "->", type(float(scores.mean())))

round(float(scores.mean()), 2), round(float(scores.std()), 2)

Two details:

- `round(x, 2)` on its own does **not** remove the label. A rounded NumPy number is still a NumPy number. Convert first, then round, as in the cell above.
- For a whole column, `.tolist()` converts every value to a plain Python number at once.

Thomas, for 2.5 and 2.6 use Fix 1 when you are writing a sentence about a number, and Fix 2 when you are showing several statistics. You will rarely need Fix 3.

#### Your Turn 5.1 - One clean sentence

Print this sentence for `orders["sales"]`, with the real values, dollars with commas and two decimals:

`Sales run from $__ to $__, with a median of $__ and a mean of $__.`

In [0]:
# Your code here
sales = orders["sales"]

print(f"Sales run from ${sales.min():,.2f} to ${sales.max():,.2f}, "
      f"with a median of ${sales.median():,.2f} and a mean of ${sales.mean():,.2f}.")

<details>
<summary><b>Show answer</b></summary>

```python
sales = orders["sales"]

print(f"Sales run from ${sales.min():,.2f} to ${sales.max():,.2f}, "
      f"with a median of ${sales.median():,.2f} and a mean of ${sales.mean():,.2f}.")
```

A sentence like that goes straight into a markdown cell or a findings memo. Notice it reports the median next to the mean, because Section 3 showed this column is skewed.

</details>

---
# 6. Functions vs methods, `sorted()` vs `.sort()`

**From Kara's post:** *"I did come back to a question I had previously and it has to do with a method vs. a function. I recall this coming up previously but I find I can't seem to remember the difference or when they are used. The example that made me stop and think is with sort. When to use the function `sorted()` and when to use the method `.sort()`."*

**From Matei's reply:** *"`sorted()` returns a new sorted list, while `.sort()` changes the existing list. Seeing them side by side would help me remember when to use each one too."*

Matei's answer is correct, and side by side is how this section does it. Kara, the reason it does not stick is that there are two separate questions tangled together here. One is about where the name lives. The other is about what happens to your data. Take them one at a time.

### 6.1 Where the name lives

- A **function** stands on its own. You call it by name and hand it the thing: `len(x)`, `sorted(x)`, `print(x)`, `round(x, 2)`.
- A **method** belongs to a type of object. You call it with a dot after the thing: `x.sort()`, `text.upper()`, `df.head()`.

A method is a function that lives inside a type. That is the whole difference, and it has one practical consequence: **a function can work on many types, but a method only exists on the types that define it.**

In [0]:
print(sorted([3, 1, 2]))             # a list
print(sorted((3, 1, 2)))             # a tuple
print(sorted("kansas"))              # a string, letter by letter
print(sorted({"b": 2, "a": 1}))      # a dictionary: its keys

`sorted()` takes anything it can loop over and always gives back a list. `.sort()` is a method of lists, and only of lists:

In [0]:
# SUPPOSED TO FAIL -- AttributeError: 'tuple' object has no attribute 'sort'
(3, 1, 2).sort()

You do not get to choose whether something is a function or a method. Whoever wrote it chose, and you look it up. Two ways to look it up:

- Type the variable name, a dot, and press `Tab`. Jupyter lists every method that object has.
- Guess, and read the error. The two errors tell you which way you guessed wrong:

| You wrote | Error | It means |
|---|---|---|
| `upper("kansas")` | `NameError: name 'upper' is not defined` | There is no function with that name. It is a method: `"kansas".upper()` |
| `[3, 1, 2].len()` | `AttributeError: 'list' object has no attribute 'len'` | This type has no such method. It is a function: `len([3, 1, 2])` |

In [0]:
"kansas".upper()

In [0]:
# SUPPOSED TO FAIL -- NameError: name 'upper' is not defined
upper("kansas")

### 6.2 What happens to your data

This is the question that actually causes bugs, and it is not really function versus method. It is: **does this hand me back a new thing, or does it change the thing I already have?**

In [0]:
readings = [7, 3, 9, 1]

result = sorted(readings)
print("sorted(readings) returned:", result)
print("readings is now:          ", readings)

In [0]:
readings = [7, 3, 9, 1]

result = readings.sort()
print("readings.sort() returned:", result)
print("readings is now:         ", readings)

| | `sorted(x)` | `x.sort()` |
|---|---|---|
| Kind | function | method of lists |
| Works on | anything you can loop over | lists only |
| Returns | a **new** sorted list | `None` |
| The original | unchanged | **reordered in place** |
| Options | `key=`, `reverse=` | `key=`, `reverse=` |

**When to use which:** use `sorted()` by default. It works on everything and it leaves your original alone, so you cannot lose anything. Use `.sort()` when you have a list, you want that list reordered, and you do not need the old order.

The one mistake to avoid is mixing the two styles. `.sort()` returns `None`, so assigning its result throws your list away. (This was in Lecture 2, and it still catches everyone once.)

In [0]:
# SUPPOSED TO FAIL -- TypeError: 'NoneType' object is not subscriptable
readings = [7, 3, 9, 1]
ranked = readings.sort()
ranked[0]

### 6.3 How to know which kind you are calling

You do not need to memorize it method by method. It follows from the type:

| Type | Can it be changed? | So its methods... | Which means you write |
|---|---|---|---|
| strings, numbers, tuples | no | always return a new value | `name = name.strip()` |
| lists, dictionaries, sets | yes | usually change it in place and return `None` (`.sort()`, `.append()`, `.reverse()`, `.update()`) | `scores.append(90)` with no `=` |
| pandas DataFrames and Series | yes | almost always return a **new** one anyway | `df = df.sort_values("sales")` |

And a test that works on anything: **run the line by itself as the last line of a cell.** If the notebook prints a result, the call handed you something new, and you need `=` to keep it. If nothing prints, it returned `None`, and it changed the original.

### 6.4 The pandas version, which is the one you need for 2.6

In pandas you sort with the method `.sort_values()`. It follows the third row of the table: it returns a new DataFrame and leaves yours alone.

In [0]:
small = orders.head(5).copy()
small

In [0]:
small.sort_values("sales")      # prints a sorted table...

In [0]:
small                           # ...but small itself has not changed

In [0]:
small.sort_values("sales", inplace = True)      # prints a sorted table...

In [0]:
small

The first cell printed a result, so by the test in 6.3 it handed back something new. If you want to keep it, assign it:

In [0]:
small = small.sort_values("sales", ascending=False)
small

This is the most common pandas mistake there is, and it has nothing to do with sorting in particular. `df.drop(...)`, `df.rename(...)`, `df.fillna(...)`, `df["col"].str.strip()`, and `df.astype(...)` all return a new object. **If you run one of them without `=`, you saw the result but you did not keep it.**

> **On `inplace=True`.** Many pandas methods accept `inplace=True`, which makes them behave like a list method: change the original, return `None`. I do not use it, and I suggest you do not either. It invites the mistake from 6.2 (`df = df.sort_values("sales", inplace=True)` leaves you with `df` equal to `None`), and it cannot be chained.

What about Python's `sorted()` on pandas objects? It runs, and it gives you something you did not want:

In [0]:
print(sorted(small))              # a DataFrame loops over its column NAMES
print(sorted(small["sales"]))     # a plain list: the row labels are gone

The first is a list of column names. The second is the sales values in order, but as a bare list with no index, so you can no longer tell which order each value came from. **In pandas, sort with `.sort_values()` or `.sort_index()`.**

### 6.5 Where sorting meets charts

One reason this matters this week: a bar chart is only readable if the bars are in order, and the order of the bars is the order of the rows. Sort, then plot.

In [0]:
totals = orders.groupby("sub_category")["sales"].sum().sort_values()

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.barh(totals.index, totals.values)
ax.set_xlabel("Total sales ($)")
ax.set_title("Total sales by sub-category, largest at the top")
plt.show()

Notice the sort is **ascending**, smallest first, and the largest bar still ends up at the top. A horizontal bar chart draws the first row at the bottom. If your `barh` chart comes out upside down in 2.6, this is why.

#### Your Turn 6.1 - Predict, then run

Write down what each of the four `print` lines will show **before** you run the cell.

In [0]:
cities = ["Wichita", "Topeka", "Salina"]

a = sorted(cities)
b = cities.sort()
c = cities[0].upper()

print(a)
print(b)
print(cities)
print(c)

---
# Where this leaves us

1. **A standard deviation is the typical distance from the mean, in the units of the data.** On its own it is never high or low.
2. **Compare it to something:** another group, a line that matters, or the mean (the CV).
3. **68-95-99.7 only works on bell-shaped data.** Draw the histogram first. If the mean and median are far apart, or the std is bigger than the mean, report the median and the IQR instead.
4. **One bad value can wreck a std**, which makes a z-score filter unreliable on small tables. The box plot fences use quartiles, which a typo cannot bend.
5. **A flag is a question, not a verdict.** Impossible values are errors. Extreme values are usually real. Never delete a row only because a rule flagged it.
6. **`np.float64(...)` is a display label, not a problem.** Use an f-string for a sentence and `.agg([...]).round(2)` for a table.
7. **Function or method is about where the name lives. New thing or changed in place is about your data**, and it follows from the type. In pandas, nearly everything returns a new object, so assign it.

---
# Assignment: Reading the Spread on the Real Superstore File

Use `Global_Superstore_Orders_2012_2015.xlsx`. Start a **new notebook** and copy in the loading and cleaning cells from your 2.6 notebook, so the names and types are already fixed. Every step below needs code **and** a markdown cell with your answer in words. No output in this notebook should show `np.float64(...)` or more than two decimals.

**1. Three numbers, plainly.** For `sales` and for `profit`, print the mean, the median, and the standard deviation in one formatted sentence each. Then in markdown, for each column: is that standard deviation high or low, and **compared to what**? Use the CV for `sales`, and explain why the CV is the wrong tool for `profit`.

**2. Test the rule.** Compute the share of `sales` values within 1, 2, and 3 standard deviations of the mean (copy `rule_check` from Section 3.1). Draw a histogram of `sales`. In markdown: why do the shares not match 68-95-99.7, and which two numbers would you report for this column instead?

**3. Flag, then decide.** Flag `sales` values two ways: `|z| > 3`, and outside the box plot fences (copy `iqr_fences` from Section 4.4). Report how many rows each rule flags. Show the ten largest orders, with enough columns to judge them. In markdown: are they errors or real orders, what would you do with them, and what does that choice cost?

**4. Compare groups.** Make one table with the mean, median, and standard deviation of `profit` by `category`, rounded to two decimals and sorted by standard deviation, largest first. In markdown: which category is the least predictable, and what does that mean for someone running that part of the business? Then one more sentence: is the thing you sorted with a function or a method, and did it change your original table? How do you know?

**Before you submit:** *Kernel > Restart Kernel and Run All Cells*, and check that it runs top to bottom. Submit the `.ipynb`.